## 1. 언어 모델의 직관적 이해

언어 모델은 문장이 얼마나 자연스러운지 확률을 구하거나, 이전 단어들이 주어졌을 때 다음 단어를 예측하는 시스템입니다. 수식 없이 직관적으로 접근해 봅니다.

스마트폰 키보드에서 **나는**이라는 단어와 **오늘**이라는 단어를 연이어 입력했을 때, 키보드는 다음 단어로 **밥을**또는 **영화를**추천합니다. 

우리의 전체 데이터가 다음과 같다고 가정합니다.
- 데이터 1: 나는 오늘 밥을 먹었다
- 데이터 2: 나는 오늘 영화를 보았다

여기서 **나는**다음 **오늘**이라는 흐름 뒤에 등장한 단어는 **밥을** 1번, **영화를** 1번입니다. 두 단어 모두 등장할 확률이 반반입니다. 이렇게 과거 단어들의 패턴을 세어서 예측하는 방식이 통계적 언어 모델입니다.

## 2. N-gram의 개념

다음 단어를 예측하기 위해 앞에 몇 개의 단어를 참고할 것인지 결정하는 것이 **N-gram**입니다.
- 1-gram (Unigram): 이전 단어를 참고하지 않고 가장 많이 나온 단어를 찍습니다.
- 2-gram (Bigram): 바로 앞 1개의 단어만 참고합니다.
- 3-gram (Trigram): 바로 앞 2개의 단어를 참고합니다.

In [1]:
sentences = [
    '나는 오늘 밥을 먹었다',
    '나는 오늘 영화를 보았다'
]
# 공백으로 분류
words_list = [sentence.split() for sentence in sentences]
words_list

[['나는', '오늘', '밥을', '먹었다'], ['나는', '오늘', '영화를', '보았다']]

In [2]:
next_words = []
for words in words_list:
    for i in range(len(words)-2):
        if words[i] == '나는' and words[i+1] == '오늘':
            next_words.append(words[i+2])
print(f'나는 오늘 다음에 등장한 단어들 : {next_words}')            

나는 오늘 다음에 등장한 단어들 : ['밥을', '영화를']


## 2. 말뭉치(Corpus)의 편향성과 토큰화(Tokenization)의 종속성

통계적 언어 모델은 철저하게 학습된 말뭉치의 세상만을 인식합니다. 
우리가 가진 전체 데이터가 다음과 같다고 가정합니다.
- 데이터 1: 컴퓨터 공학은 매우 흥미롭다
- 데이터 2: 컴퓨터 공학은 취업이 잘된다

이 세상에서 **컴퓨터**다음 **공학은**이라는 흐름이 나왔을 때, 이 모델은 세상에 존재하는 다음 단어가 오직 **매우**혹은 **취업이**두 가지만 존재한다고 확신합니다. 만약 입력으로 **컴퓨터 공학은 어렵다**라는 문장이 들어오면, 이 모델은 확률을 0으로 만들어버립니다. 이를 통해 우리는 양질의 거대한 말뭉치 구축이 통계적 모델링의 핵심임을 알 수 있습니다.

또한, N-gram 모델은 텍스트를 자르는 단위인 토큰화 방식에 절대적으로 의존합니다.
**나는 오늘 밥을 먹었다**문장을 어절(띄어쓰기) 단위로 자르면 다음과 같습니다.
- [나는, 오늘, 밥을, 먹었다] -> 총 4개의 토큰

반면, 형태소 단위로 자르면 다음과 같이 변합니다.
- [나, 는, 오늘, 밥, 을, 먹, 었, 다] -> 총 8개의 토큰

이때 Bigram(바로 앞 1개 단어 참고)을 구축한다고 하면, 어절 단위에서는 **오늘**다음 **밥을**이 연결되지만, 형태소 단위에서는 **오늘**다음 **밥**이 연결됩니다. 한국어의 경우 교착어의 특성상 띄어쓰기 기반 N-gram은 조사의 결합에 따라 완전히 다른 패턴으로 분산되어 성능이 극도로 저하되므로, 형태소 분석이 필수적으로 동반되어야 함을 직관적으로 이해할 수 있습니다.

## 3. N-gram의 한계: 장기 의존성(Long-term Dependency) 문제

앞의 단어를 참고하여 다음을 예측하는 N-gram은 구조적으로 치명적인 맹점을 가집니다. 바로 문맥이 길어질 때 발생하는 장기 의존성 문제입니다.

- 예시 문장: "그 프랑스 출신 요리사는 서울에 온 지 10년이 넘었지만 여전히 아침마다 고향을 그리워하며 **[?]** 로 중얼거린다"

사람은 이 문장을 읽고 주어가 **프랑스 출신 요리사** 라는 아주 먼 과거의 정보를 기억하여 괄호 안에 **프랑스어** 가 들어갈 확률이 높다고 추론합니다.
하지만 일반적인 통계 모델이 사용하는 Trigram(앞의 2개 단어 참고) 구조에서는 오직 **고향을** 과 **그리워하며** 라는 두 단어만 보고 다음 단어를 찍어야 합니다. 앞의 **프랑스** 라는 정보는 모델의 시야에서 완전히 삭제되었기 때문에 제대로 된 예측이 불가능해집니다. 이 한계를 극복하기 위해 N을 무한정 늘리면, 똑같은 패턴을 과거 데이터에서 단 한 번도 찾지 못하는 희소 문제에 직면하게 됩니다.

## 4. 특수 토큰의 도입: 시작(BOS)과 종료(EOS)

단어와 단어의 연결성만 세다 보면, 문장이 어디서 시작되고 어디서 끝나는지 모델은 알 수 없습니다. 이를 위해 데이터 앞뒤에 특수한 가짜 단어를 삽입해야 합니다.
- `<s>` (Begin of Sentence): 문장의 시작을 알림
- `</s>` (End of Sentence): 문장의 끝을 알림

기존: 나는 밥을 먹었다
전처리 후: `<s>` 나는 밥을 먹었다 `</s>`

이렇게 전처리하면, 모델은 `<s>` 다음에 **나는** 이 올 확률을 계산함으로써 문장의 첫 단어로 어떤 단어들이 주로 등장하는지 통계적으로 학습할 수 있게 됩니다. 마찬가지로 **먹었다** 다음에 `</s>` 가 올 패턴을 통해 문장을 안전하게 끝마치는 법을 터득합니다.

## 1. 말뭉치 특수 토큰 부착 및 기본 토큰화
문장의 시작을 의미하는 `<s>` 와 끝을 의미하는 `</s>` 를 부착하여 모델이 문장의 경계를 인식할 수 있도록 데이터를 정제합니다.

In [3]:
raw_corpus = [
    '나는 오늘 파이토치 학습을 한다',
    '나는 내일 자연어 처리를 한다',
    '오늘 파이토치 학습은 매우 즐겁다',
    '파이토치 코딩은 매우 재미있다'
]
# BOS(Begin of Sequence) EOS(End of Sequence) 부착 함수
def process_corpus(corpus):
    processed = []
    for sentence in corpus:
        processed.append(['<s>']  + sentence.split() + ['</s>'] )
    return processed

tokenized_corpus = process_corpus(raw_corpus)
tokenized_corpus

[['<s>', '나는', '오늘', '파이토치', '학습을', '한다', '</s>'],
 ['<s>', '나는', '내일', '자연어', '처리를', '한다', '</s>'],
 ['<s>', '오늘', '파이토치', '학습은', '매우', '즐겁다', '</s>'],
 ['<s>', '파이토치', '코딩은', '매우', '재미있다', '</s>']]

## 2. N-gram 추출기 구현 및 데이터베이스화
Python의 `collections` 모듈과 제너레이터 방식을 활용하여, N의 크기에 상관없이 유연하게 N-gram 조각들을 추출하고 빈도를 카운트하는 범용 알고리즘을 설계합니다.

In [4]:
from collections import Counter
def extract_ngrams(tokenized_corpus, n):
    ngram_counts = Counter()
    for tokens in tokenized_corpus:
        # 토큰길이가 N보다 작으면 추출 불가
        if len(tokens) < n:
            continue
        # 슬라이딩 윈도우 방식
        for i in range(len(tokens) - n+1):
            ngram = tuple(tokens[i : i + n])
            ngram_counts[ngram] += 1
    return ngram_counts

In [5]:
# unigram(1-gram) 추출
unigram = extract_ngrams(tokenized_corpus,1)
bigram = extract_ngrams(tokenized_corpus,2)

In [6]:
unigram.most_common(5), bigram.most_common(5)

([(('<s>',), 4), (('</s>',), 4), (('파이토치',), 3), (('나는',), 2), (('오늘',), 2)],
 [(('<s>', '나는'), 2),
  (('오늘', '파이토치'), 2),
  (('한다', '</s>'), 2),
  (('나는', '오늘'), 1),
  (('파이토치', '학습을'), 1)])

## 3. 문맥(Context) 기반 다음 단어 예측 시뮬레이션
추출된 N-gram 데이터베이스를 활용하여, 특정한 앞 단어(Context)가 주어졌을 때 뒤이어 올 단어들의 분포를 확인하여 통계적 언어 모델의 원형을 모사합니다.

In [7]:
def simulate_prediction(ngram_counts, context_words):
    # context_words : 뒤에올 단어 예측 후보
    context_len = len(context_words)
    found = False
    # 튜플의 앞부분이 context_words와 일치하는 n-gram을 탐색
    for ngram, count in ngram_counts.items():
        if list(ngram[:context_len]) == context_words:
            next_word = ngram[context_len]
            print(f'-->예측단어 : {next_word} 등장횟수 : {count}번')
            found = True
    if not found:
        print(f'--> 데이터베이스에 해당 문맥이 존재하지 않습니다(희소문제발생)')

In [8]:
# 나는 다음에 올 단어 예측(Bigram, 2-gram) 
simulate_prediction(bigram, ['나는'])

-->예측단어 : 오늘 등장횟수 : 1번
-->예측단어 : 내일 등장횟수 : 1번


In [10]:
# Trigram
trigram = extract_ngrams(tokenized_corpus,3)
simulate_prediction(trigram, ['나는','오늘'])

-->예측단어 : 파이토치 등장횟수 : 1번


In [11]:
# 희소문제 데이터에 없는 패턴 테스트
simulate_prediction(trigram, ['파이토치','처리를'])

--> 데이터베이스에 해당 문맥이 존재하지 않습니다(희소문제발생)


In [13]:
four_gram = trigram = extract_ngrams(tokenized_corpus,4)
simulate_prediction(four_gram, ['나는','오늘','파이토치'])

-->예측단어 : 학습을 등장횟수 : 1번
